# Embeddings & Evidence Retrieval

A fluent answer can hide missing evidence. To diagnose an AI system, follow the path from **source material → representation → retrieval → context → answer**. Each stage can fail for a different reason.

## What an Embedding Represents

Embedding
: A numerical representation of an input, used to compare it with other inputs in a learned space. The vector as a whole represents the passage; individual coordinates are not normally named words or human-readable concepts.

$$e=f_{\theta}(x)\in\mathbb{R}^{d}.$$

Here $x$ is the input passage, $\theta$ denotes learned model parameters, $e$ is its embedding, and $d$ is the number of coordinates. The numbers are useful through their relationships to other representations.

:::{admonition} The Apple example
:class: tip ds-example-card

“Apple released a laptop” and “slice the apple” provide different context and can receive different representations. The isolated word “Apple” remains ambiguous. A diagram with a fruit axis and a device axis is only a teaching analogy: real learned coordinates are generally not so interpretable.

Two passages can also be similar while contradicting one another, because they discuss the same subject. Similarity is not a truth test.
:::

```{figure} figures/vector-lookup.svg
:alt: A passage is indexed as vector C17; search returns identifier C17 and lookup returns the stored text.

The vector locates the evidence. The original text is recovered by identifier, not reconstructed by reversing the embedding.
```

Store a passage's text, or a usable reference to it, alongside its identifier and vector. At query time, encode the question, rank passage vectors, and look up the text for the returned identifiers. Losing provenance makes a good search score difficult to use or verify.

Chunking
: Dividing source material into retrievable units while preserving the meaning needed to answer questions.

A chunk containing only “timeout: 5 seconds” may omit the service and conditions. A whole manual can contain too much unrelated text. Headings and coherent paragraphs help preserve context; overlap can protect boundaries but create duplicates that need deduplication.


## How Retrieval Training Works

One common training pattern pairs a query with a relevant passage and less relevant passages. The encoder learns to give the relevant pair a higher score. This is one approach, not the recipe for every embedding model.

```{figure} figures/retrieval-training.svg
:alt: Training examples are encoded, a ranking loss is calculated, and encoder parameters are updated.

Repeated training changes the representation. Searching an already indexed collection does not retrain the encoder on each question.
```

For “Why does login expire early?”, a passage about session timeout is a positive example. A passage about login-page styling is a useful **hard negative**: its vocabulary overlaps, but it does not answer the question.

A simplified contrastive loss is

$$\mathcal{L}=-\log\frac{\exp(s(q,p^+)/\tau)}{\exp(s(q,p^+)/\tau)+\sum_j\exp(s(q,p_j^-)/\tau)}.$$

| Symbol | Meaning |
|---|---|
| $q$ | Query embedding |
| $p^+$ | Relevant passage embedding |
| $p_j^-$ | The $j$-th negative passage embedding |
| $s$ | Similarity score, such as cosine similarity |
| $\tau>0$ | Temperature controlling how sharply scores are distinguished |

The fraction is the positive passage's share of the exponentiated scores. Minimizing the negative logarithm encourages that share to increase.

:::{admonition} Compatibility is more than equal vector length
:class: warning

Query and passage representations must belong to a compatible space, with appropriate preprocessing, normalization, and query/document settings. Using one encoder version is common; co-trained query and passage encoders can also be compatible. Two unrelated models producing the same number of coordinates are not thereby compatible.
:::

Both query and passage processing are usually **encoding** operations. The downstream generator receives readable text, so it can be changed independently of the embedding model. Changing the embedding space usually requires rebuilding document vectors and switching the query encoder with the index.


## Evaluate Retrieval Separately

Top-$k$
: The number of ranked candidates requested from retrieval. It is a setting, not itself a measure of quality.

Let $R_q$ be the known relevant chunks for query $q$, and $S_k(q)$ the first $k$ retrieved chunks. Assuming $k$ results are returned,

$$\operatorname{Precision@}k=\frac{|S_k(q)\cap R_q|}{k},\qquad
\operatorname{Recall@}k=\frac{|S_k(q)\cap R_q|}{|R_q|}.$$

Precision asks how much of the returned material is relevant. Recall asks how much of the known relevant material was found. Recall requires at least one known relevant item; incomplete relevance labels also limit what the score can tell us.

Reciprocal rank
: $1/r$, where $r$ is the position of the first relevant result, or zero when none is found within the evaluated list. Mean reciprocal rank averages this value over queries.

:::{admonition} A ranked list: A, B, C, D
:class: tip ds-example-card

Suppose **B and D are the only relevant chunks**.

| Rank | Chunk | Relevant? | What changes? |
|---|---|---|---|
| 1 | A | No | No relevant evidence yet |
| 2 | B | Yes | First relevant result; reciprocal rank $=1/2$ |
| 3 | C | No | Precision falls while recall stays unchanged |
| 4 | D | Yes | Both relevant chunks are now available |

At $k=3$: precision $=1/3$, recall $=1/2$, reciprocal rank $=1/2$.

At $k=4$: precision $=2/4=1/2$, recall $=2/2=1$, reciprocal rank remains $1/2$.

If the answer needs **both** B and D, finding only B is insufficient even though the first relevant result appears early. Increasing $k$ improves coverage in this example but also consumes more reading and context capacity.
:::

::::{div}
:class: reading-comparison

:::{div}
### Retrieval Evaluation

Use representative questions with labeled evidence. Compare ranking and coverage while controlling chunking, filters, and search settings when isolating the encoder's effect.
:::

:::{div}
### Answer Evaluation

Check correctness, evidence support, source citations, and appropriate uncertainty. Correct evidence can be ignored, and a plausible answer can conceal a retrieval failure.
:::
::::


## Improve the Stage That Failed

| Intervention | What changes | What can go wrong |
|---|---|---|
| Query rewriting | Search wording | The rewritten query distorts intent |
| Reranking | Candidate order after initial retrieval | Extra latency; missing candidates cannot be recovered by reordering alone |
| Deduplication | Space used by repeated evidence | Overaggressive merging removes distinct qualifications |
| Version filters | Which revisions are eligible | The needed revision is excluded |
| Access filters | Which evidence the user may use | Stale permissions expose or hide information incorrectly |
| Better chunking | Meaning retained in each candidate | Boundaries separate a claim from its conditions |

These operations need not change the encoder. Measure their contribution separately. A public benchmark can help screen models, but internal acronyms, code, permissions, and document versions require evaluation on the actual collection.

:::{admonition} Bigger is not a quality guarantee
:class: note

Compare retrieval quality, ingestion speed, query latency, memory, and deployment constraints. CPU feasibility depends on throughput and latency requirements. A larger model or a better unrelated benchmark score does not guarantee better results on the questions this system must answer.
:::


## Keep the Index Current

Freshness
: Whether retrieved evidence still matches the authoritative source and the version relevant to the question.

Keep a stable source identifier, source version, chunk identifier, and content hash. A hash acts as a fingerprint for detecting content changes. Update stored text, vectors, and metadata consistently.

| Source state | Refresh action | Why it matters |
|---|---|---|
| Unchanged | Reuse the embedding | Avoid unnecessary recomputation |
| Added or changed | Identify affected chunks, re-chunk, and embed | Keep text and vectors aligned |
| Deleted | Remove obsolete records | Prevent retrieval of withdrawn content |
| Access changed | Restrict or remove as required | Old indexing does not preserve authorization |
| Embedding model changed | Rebuild compatible vectors and switch query encoder together | Avoid comparisons across incompatible spaces |

:::{admonition} One paragraph can affect several chunks
:class: note

An edit may shift chunk boundaries or appear in several overlapping windows. Updating a single vector is safe only if that really covers the affected region. Publish a consistent searchable version instead of mixing old text, new vectors, and stale metadata.
:::

Direct file search also has freshness limits: searching an old mirror returns old evidence. A live service can have caches or indexing delays. Compare retrieved revisions or timestamps with authoritative sources when freshness matters.


## Direct Search, RAG and MCP

Direct tool-based search
: Investigating files or services at query time by searching for terms, reading selected results, and refining the search based on what is found.

| Method or layer | Useful contribution | Limitation |
|---|---|---|
| Keyword / exact search | Error codes, filenames, identifiers | Vocabulary mismatch can hide evidence |
| Semantic retrieval | Paraphrases and conceptual similarity | Similarity can miss decisive distinctions |
| Knowledge graph traversal | Explicit relationships, such as service → dependency → owner | Stale or incomplete edges can misdirect the search |
| MCP / another API | Exposes a capability to a client | Does not determine the underlying search algorithm |

Direct search and embedding retrieval can both be exposed through MCP. A server may provide keyword search, vector search, a document lookup, or an assistant that internally performs several steps. The interface and the retrieval method answer different architectural questions.

:::{admonition} Labels from the discussion are not universal results
:class: warning

The supplied paper reports “DCI” as an uncertain name for direct tool-based search; its expansion and referenced article were not established. Claims about a universal document threshold or 20–30× token multiplier were also unverified. Search cost depends on the collection, tool efficiency, question, and interaction count. These notes do not treat those claims as facts.
:::


## Worked Investigation: Session Rejection

```{figure} figures/evidence-check.svg
:alt: Exact and semantic evidence are combined, checked for version and access, and used to produce a cited answer.

Exact search supplies incident-specific facts; semantic search supplies explanatory context. The two can run independently before their evidence is combined.
```

:::{admonition} Why does service A reject sessions after deployment?
:class: tip ds-example-card

This hypothetical example uses illustrative identifiers.

1. **Exact evidence:** search the current log for error E42 and inspect the timeout configuration diff.
2. **Explanatory evidence:** retrieve the design passage about session lifetime and the authentication cache.
3. **Verification:** check the passage version and read the deployed setting from the authoritative service.
4. **Context assembly:** merge candidates by source identity, remove duplicates, and retain neighboring qualifications.
5. **Answer:** connect the observed rejection to the supported mechanism, cite the log and design passage, and separate a causal hypothesis from a verified cause.

If the deployed value cannot be checked, state that limitation instead of presenting the configuration explanation as established fact.
:::

| Observed failure | Component to investigate first |
|---|---|
| E42 is never found | Search terms, source access, coverage |
| An obsolete design passage is returned | Refresh process and version filters |
| Correct evidence is supplied but contradicted | Context assembly and generation |
| Citation points to the wrong chunk | Identifier and provenance handling |

One end-to-end score cannot identify which component needs repair. Test the evidence path stage by stage and then test the complete answer.
